# Rotating Bar Example

A simple work-through example to model the effect of a rotating bar on the Milky Way potential.


In [13]:
import logging

import galax.potential as gp
import numpy as np
import optax
import unxt as u
from flax import nnx

from galactoPINNs.data import (
    flatten_time_dict_by_time,
    generate_time_dep_data,
    scale_data_time,
)
from galactoPINNs.evaluate import evaluate_performance_node
from galactoPINNs.models.node_model import NODEModel
from galactoPINNs.train import train_model_node

## Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

In [14]:
####
### Generate synthetic data for training and testing
####

MW_potential = gp.MilkyWayPotential(units="galactic")
true_rotation_rate = 2 # degree per Myr

def alpha_of_t(t: u.Quantity["time"]) -> u.Quantity["angle"]:
    t_myr = t.to_value("myr")
    return u.Quantity(t_myr * true_rotation_rate, "degree")

rot_bar = gp.LongMuraliBarPotential(
            m_tot=u.Quantity(1e10, "Msun"), a=u.Quantity(4.0, "kpc"), b=u.Quantity(1.0, "kpc"), c=u.Quantity(1.5, "kpc"),
            alpha = alpha_of_t, units="galactic")

mw_lmc_bar = MW_potential + rot_bar

true_analytic_function = mw_lmc_bar
analytic_baseline_potential = rot_bar

In [15]:
####
### Generate the training and testing sets
####

N_samples_train = 1024
N_samples_test = 1024
r_max_train = 150 #kpc
r_max_test = 200  #kpc

ts_train = np.linspace(0, 100, 6)
ts_test = np.linspace(0, 180, 8)
times_train = ts_train
times_test = ts_test

raw_datadict = generate_time_dep_data(
    galax_potential= true_analytic_function,
    times_train = times_train,
    times_test = times_test,
    n_samples_train= N_samples_train,
    n_samples_test= N_samples_test,
    r_max_train =100,    # kpc
    r_max_test  =150,    # kpc
)


In [16]:
####
### Nondimensionalize the data, and set up the model configuration
####

include_analytic = True
scale = "nfw"
r_s = 15.62 #kpc

initial_config = {
    "r_s": r_s,
    "include_analytic": include_analytic,
    "ab_potential": analytic_baseline_potential}

# nondimensionalize the data
data, transformers = scale_data_time(
    raw_datadict, initial_config)

# configure the desired model features
config = {
    "x_transformer": transformers["x"],
    "a_transformer": transformers["a"],
    "u_transformer": transformers["u"],
    "t_transformer": transformers["t"],
    "r_s": r_s,
    "scale": scale,
    "enforce_bc": False,
    "include_analytic": include_analytic,
    "delta_phi_depth": 3,
    "delta_phi_width": 64,
    "initial_correction_depth": 4,
    "integration_mode": "gl3"
    }


In [17]:
###
## Initialize and train a model with a non-trainable baseline potential
###

## set up the model and hyperparameters
l_rel = 0.5  # weight for the relative loss term
lr0 = 1e-3  # initial learning rate
tx = optax.adam(lr0)
net = NODEModel(config,rngs=nnx.Rngs(0))

## set up the optimizer
opt = nnx.Optimizer(net, optax.adam(lr0), wrt=nnx.Param)

x_train, a_train = flatten_time_dict_by_time(data, split="train")

In [18]:
###
## Train the model
## Should converge in < 2 minutes. For optimal results, train longer
###

out = train_model_node(
    model=net,
    optimizer=opt,
    x_train = x_train,
    a_train = a_train,
    num_epochs = 2000,
    log_every = 100,
)

2026-05-25 19:48:57,389 | INFO | galactoPINNs.train | Epoch 0, Loss: 1.055400
2026-05-25 19:49:01,668 | INFO | galactoPINNs.train | Epoch 100, Loss: 0.029917
2026-05-25 19:49:07,039 | INFO | galactoPINNs.train | Epoch 200, Loss: 0.022951
2026-05-25 19:49:11,826 | INFO | galactoPINNs.train | Epoch 300, Loss: 0.019101
2026-05-25 19:49:16,981 | INFO | galactoPINNs.train | Epoch 400, Loss: 0.015468
2026-05-25 19:49:22,207 | INFO | galactoPINNs.train | Epoch 500, Loss: 0.013597
2026-05-25 19:49:29,307 | INFO | galactoPINNs.train | Epoch 600, Loss: 0.012708
2026-05-25 19:49:34,571 | INFO | galactoPINNs.train | Epoch 700, Loss: 0.011332
2026-05-25 19:49:40,162 | INFO | galactoPINNs.train | Epoch 800, Loss: 0.010606
2026-05-25 19:49:45,671 | INFO | galactoPINNs.train | Epoch 900, Loss: 0.010448
2026-05-25 19:49:50,719 | INFO | galactoPINNs.train | Epoch 1000, Loss: 0.009577
2026-05-25 19:49:55,769 | INFO | galactoPINNs.train | Epoch 1100, Loss: 0.008794
2026-05-25 19:50:00,876 | INFO | galacto

In [19]:
###
## Evaluate performance
###

perf = evaluate_performance_node(
    model=out["model"],
    t_eval = times_test[-1],
    raw_datadict = raw_datadict,
    num_test = 800,
)
